# Enhanced Scalability of Horseshoe-and-Spur Networks by Exploiting Hollow-Core Fiber
## Physics/optimization-based reproduction of Figs. 2–5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/paper/Horseshoe/horseshoe_fig2_fig3_fig4_fig5_reproduction.ipynb)

- Paper: [arXiv:2608.07082](https://arxiv.org/abs/2608.07082)
- Authors: Mohammad M. Hosseini, João Pedro, Antonio Napoli
- Target: reproduce the *shape and physical trends* of Figs. 2, 3, 4 and 5 without inserting the plotted curve coordinates.

**중요한 재현 범위**

Fig. 2와 Fig. 5는 논문에 공개된 식으로 직접 계산합니다. Fig. 3과 Fig. 4는 저자들의 2024년 논문에 정의된 전체 ILP와 실제 10개 네트워크 표본이 공개되어 있지 않으므로, 공개된 제약을 반영한 재현 가능한 MILP surrogate를 사용합니다. 따라서 곡선의 추세·상한·포화 현상은 재현되지만 각 점은 논문과 다를 수 있습니다. 논문 그래프의 좌표는 계산 입력이나 보정값으로 사용하지 않습니다.

## 1. 모델 유도

### Fig. 2 — 증폭기 총출력으로부터 subcarrier 출력 상한 계산

400G DSCM 채널 하나는 16개 subcarrier(SC)를 사용하므로 선형 전력에서

$$16N_{Ch}P_{SC}^{out}\le P_A.$$

이를 dBm으로 쓰면

$$P_{SC,max}^{out}[\mathrm{dBm}]=P_A[\mathrm{dBm}]-10\log_{10}(16N_{Ch}).$$

Fig. 3/4의 실제 fiber launch 상한은 위 총출력 상한과 비선형 임계값 중 작은 값입니다.

$$P_{cap}=\min\{P_{SC,max}^{out},P_{NL}\},\quad
P_{NL,SCF}=-8\ \mathrm{dBm},\quad P_{NL,HCF}=10\ \mathrm{dBm}.$$

### Fig. 3/4 — 공개 제약 기반 MILP surrogate

5개 transit node와 양방향 horseshoe를 구성하고 다음을 동시에 최적화합니다.

1. 각 main-link amplifier의 배치와 0–20 dB gain
2. 각 spur downstream amplifier의 배치
3. balanced 50:50 또는 unbalanced 90:10–10:90 coupler 선택
4. 수신감도 $-24$ dBm, 최대 power imbalance 8 dB, amplifier 출력/비선형 상한
5. uniform 경우 모든 spur가 같은 budget, non-uniform 경우 평균 budget 최대화

링크 길이는 선행 논문에 공개된 평균 12 km, 표준편차 11.3 km를 갖는 log-normal surrogate에서 층화 표본추출합니다. 고정 seed를 사용하므로 매 실행 결과가 같습니다. Full-HCF는 horseshoe와 spur가 HCF이고, hybrid는 horseshoe가 SCF이며 spur만 HCF입니다.

### Fig. 5 — passive spur 식

$a=10^{-\alpha L/10}$, $\alpha=0.24$ dB/km이면

$$P_{b,single}=\alpha L+10\log_{10}N,$$

$$P_{b,multi}=N\alpha L+10(N-1)\log_{10}2.$$

In [ ]:
import math
import os
import warnings
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import lil_matrix
from scipy.stats import lognorm, t

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({
    "figure.dpi": 125,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "font.size": 10,
})

# Colab에서 N_JOBS=2가 안정적입니다. 메모리가 부족하면 1로 변경하십시오.
N_JOBS = int(os.environ.get("HORSESHOE_N_JOBS", "2"))
RANDOM_SEED = 260807082
# Colab default is the paper's 10-network ensemble. The environment override is
# useful only for a quick smoke test (for example HORSESHOE_N_NETWORKS=1).
N_NETWORKS = int(os.environ.get("HORSESHOE_N_NETWORKS", "10"))
AMP_COUNTS = np.arange(4, 17)


@dataclass(frozen=True)
class Parameters:
    n_nodes: int = 5
    launch_dbm: float = -12.0
    sensitivity_dbm: float = -24.0
    alpha_db_per_km: float = 0.24
    coupler_excess_db: float = 0.5
    hcf_splice_db: float = 0.2
    max_amp_gain_db: float = 20.0
    max_imbalance_db: float = 8.0
    # Effective second passive stage of the filterless transit-node surrogate:
    # one 50:50 stage (3.01 dB) plus about 0.5 dB excess loss.
    effective_node_stage_db: float = 3.5


P = Parameters()


def aggregate_sc_limit_dbm(n_ch, p_amp_dbm):
    """Fig. 2: amplifier aggregate-output constraint, converted to per-SC dBm."""
    return np.asarray(p_amp_dbm) - 10.0 * np.log10(16.0 * np.asarray(n_ch))


def per_sc_cap_dbm(fiber, n_ch=10, p_amp_dbm=27):
    """Minimum of aggregate amplifier limit and fiber nonlinear threshold."""
    aggregate = float(aggregate_sc_limit_dbm(n_ch, p_amp_dbm))
    nonlinear = -8.0 if fiber == "SCF" else 10.0
    return min(aggregate, nonlinear)


def make_network_ensemble(n_networks=N_NETWORKS, seed=RANDOM_SEED):
    """Six-link horseshoes with moment-matched, stratified log-normal lengths."""
    n_links = 6 * n_networks  # hub–5 transit nodes–hub
    target_mean, target_std = 12.0, 11.3
    sigma = np.sqrt(np.log1p((target_std / target_mean) ** 2))
    mu = np.log(target_mean) - 0.5 * sigma**2
    quantiles = (np.arange(n_links) + 0.5) / n_links
    lengths = lognorm.ppf(quantiles, s=sigma, scale=np.exp(mu))
    lengths *= target_mean / lengths.mean()
    rng = np.random.default_rng(seed)
    rng.shuffle(lengths)
    return lengths.reshape(n_networks, 6)


NETWORKS = make_network_ensemble()
print(
    f"Synthetic ensemble: mean={NETWORKS.mean():.2f} km, "
    f"std={NETWORKS.std(ddof=0):.2f} km, "
    f"range={NETWORKS.min():.2f}–{NETWORKS.max():.2f} km"
)

## 2. Fig. 2 재현

아래 heatmap의 값은 단순히 논문 그림의 색을 흉내 낸 값이 아니라, 위 DSCM 총출력 제약식을 모든 $(P_A,N_{Ch})$ 조합에 대해 직접 계산한 결과입니다.

In [ ]:
p_a = np.linspace(26, 38, 121)
n_ch = np.linspace(6, 23, 120)
PA, NCH = np.meshgrid(p_a, n_ch)
P_SC = aggregate_sc_limit_dbm(NCH, PA)

fig, ax = plt.subplots(figsize=(7.0, 4.7))
mesh = ax.pcolormesh(PA, NCH, P_SC, shading="auto", cmap="turbo", vmin=-7, vmax=24)
cb = fig.colorbar(mesh, ax=ax, pad=0.05)
cb.set_label(r"$P_{SC}^{out}$ [dBm]")

selected = [(27, 20), (37, 20), (27, 10), (37, 10)]
for pa_i, nch_i in selected:
    value = float(aggregate_sc_limit_dbm(nch_i, pa_i))
    ax.plot(pa_i, nch_i, "o", ms=6, mfc="none", mec="#333333", mew=1.8)
    dx = 0.15 if pa_i < 32 else -2.7
    ax.text(pa_i + dx, nch_i + 1.0, f"{value:.1f} dBm", weight="bold")

ax.set(xlim=(26, 38), ylim=(6, 23), xlabel=r"$P_A$ [dBm]", ylabel=r"$N_{Ch}$")
ax.set_title("Fig. 2 reproduction — per-subcarrier amplifier output limit")
plt.tight_layout()
plt.show()

## 3. Fig. 3/4용 MILP

dBm 영역의 power flow는 덧셈식이므로 선형 제약으로 표현할 수 있습니다. Binary variable은 amplifier 배치와 discrete coupler ratio를 나타냅니다. 증폭기 출력은 입력 이상, 입력+20 dB 이하, 그리고 $P_{cap}$ 이하로 제한됩니다.

In [ ]:
class MilpBuilder:
    def __init__(self):
        self.names, self.lb, self.ub, self.integrality, self.rows = [], [], [], [], []

    def var(self, name, lb=-100.0, ub=100.0, integer=0):
        idx = len(self.names)
        self.names.append(name)
        self.lb.append(lb)
        self.ub.append(ub)
        self.integrality.append(integer)
        return idx

    def add(self, terms, lb=-np.inf, ub=np.inf):
        self.rows.append((dict(terms), lb, ub))

    def eq(self, terms, rhs):
        self.add(terms, rhs, rhs)

    def solve(self, objective):
        n_var, n_row = len(self.names), len(self.rows)
        A = lil_matrix((n_row, n_var), dtype=float)
        lower, upper = np.empty(n_row), np.empty(n_row)
        for row, (terms, lb, ub) in enumerate(self.rows):
            for col, coefficient in terms.items():
                A[row, col] = coefficient
            lower[row], upper[row] = lb, ub
        c = np.zeros(n_var)
        for col, coefficient in objective.items():
            c[col] = -coefficient  # scipy.milp minimizes
        result = milp(
            c,
            integrality=np.asarray(self.integrality),
            bounds=Bounds(self.lb, self.ub),
            constraints=LinearConstraint(A.tocsr(), lower, upper),
            options={"time_limit": 12.0, "mip_rel_gap": 2e-4, "presolve": True},
        )
        values = None if result.x is None else {
            name: result.x[i] for i, name in enumerate(self.names)
        }
        return result, values


def optimize_spur_budget(
    link_lengths,
    amp_budget,
    fiber="HCF",
    n_ch=10,
    p_amp_dbm=27,
    couplers="balanced",
    allocation="uniform",
    hybrid=False,
):
    """MILP surrogate: maximize downstream spur power budget for one network."""
    n = P.n_nodes
    model = MilpBuilder()

    main_fiber = "SCF" if hybrid else fiber
    spur_fiber = "HCF" if hybrid else fiber
    cap_main = per_sc_cap_dbm(main_fiber, n_ch, p_amp_dbm)
    cap_spur = per_sc_cap_dbm(spur_fiber, n_ch, p_amp_dbm)

    # A 90:10 coupler can be physically oriented either way, hence 10%–90%.
    ratios = [0.5] if couplers == "balanced" else np.arange(0.1, 1.0, 0.1).tolist()
    through_loss = [
        -10 * math.log10(r) + P.coupler_excess_db + P.effective_node_stage_db
        for r in ratios
    ]
    drop_loss = [
        -10 * math.log10(1 - r) + P.coupler_excess_db + P.effective_node_stage_db
        for r in ratios
    ]

    main_amp, q, through, drop_in, spur_out, ratio_choice = {}, {}, {}, {}, {}, {}
    # The downstream spur booster is represented once per spur; upstream spur
    # amplification is outside the downstream power-budget objective.
    spur_amp = {node: model.var(f"yd_{node}", 0, 1, 1) for node in range(n)}

    for direction in range(2):
        previous_through = None
        for order in range(n):
            node = order if direction == 0 else n - 1 - order
            segment = order if direction == 0 else n - order
            segment_loss = P.alpha_db_per_km * float(link_lengths[segment])
            if main_fiber == "HCF":
                segment_loss += P.hcf_splice_db

            ym = model.var(f"ym_{direction}_{node}", 0, 1, 1)
            main_amp[direction, node] = ym
            qj = model.var(f"q_{direction}_{node}", -90, cap_main)
            tj = model.var(f"t_{direction}_{node}", -110, cap_main)
            dj = model.var(f"din_{direction}_{node}", -110, cap_main)
            sj = model.var(f"s_{direction}_{node}", -110, cap_spur)
            q[direction, node], through[direction, node] = qj, tj
            drop_in[direction, node], spur_out[direction, node] = dj, sj

            if previous_through is None:
                input_terms, input_constant = {}, P.launch_dbm - segment_loss
            else:
                input_terms, input_constant = {previous_through: 1.0}, -segment_loss

            # q >= p_in and q <= p_in + Gmax*y_amp; q upper bound enforces cap.
            model.add(
                {qj: 1.0, **{idx: -coef for idx, coef in input_terms.items()}},
                lb=input_constant,
            )
            gain_terms = {qj: 1.0, ym: -P.max_amp_gain_db}
            for idx, coef in input_terms.items():
                gain_terms[idx] = gain_terms.get(idx, 0.0) - coef
            model.add(gain_terms, ub=input_constant)

            choices = []
            for ir in range(len(ratios)):
                z = model.var(f"z_{direction}_{node}_{ir}", 0, 1, 1)
                ratio_choice[direction, node, ir] = z
                choices.append(z)
            model.eq({z: 1.0 for z in choices}, 1.0)

            eq_through = {tj: 1.0, qj: -1.0}
            eq_drop = {dj: 1.0, qj: -1.0}
            for ir, z in enumerate(choices):
                eq_through[z] = through_loss[ir]
                eq_drop[z] = drop_loss[ir]
            model.eq(eq_through, 0.0)
            model.eq(eq_drop, 0.0)

            spur_splice = P.hcf_splice_db if spur_fiber == "HCF" else 0.0
            model.add({sj: 1.0, dj: -1.0}, lb=-spur_splice)
            model.add(
                {sj: 1.0, dj: -1.0, spur_amp[node]: -P.max_amp_gain_db},
                ub=-spur_splice,
            )
            previous_through = tj

    # One physical coupler ratio per transit node, shared by both directions.
    if len(ratios) > 1:
        for node in range(n):
            for ir in range(len(ratios)):
                model.eq(
                    {ratio_choice[0, node, ir]: 1.0, ratio_choice[1, node, ir]: -1.0},
                    0.0,
                )

    all_amplifiers = list(main_amp.values()) + list(spur_amp.values())
    model.add({idx: 1.0 for idx in all_amplifiers}, ub=float(amp_budget))

    budgets = []
    for node in range(n):
        bj = model.var(f"budget_{node}", 0.0, 60.0)
        budgets.append(bj)
        for direction in range(2):
            # Both hub directions must be able to support the allocated spur budget.
            model.add(
                {bj: 1.0, spur_out[direction, node]: -1.0},
                ub=-P.sensitivity_dbm,
            )

    if allocation == "uniform":
        common = model.var("budget_common", 0.0, 60.0)
        for bj in budgets:
            model.eq({bj: 1.0, common: -1.0}, 0.0)
        objective = {common: 1.0}
    else:
        for i in range(n):
            for j in range(i + 1, n):
                model.add({budgets[i]: 1.0, budgets[j]: -1.0}, ub=P.max_imbalance_db)
                model.add({budgets[j]: 1.0, budgets[i]: -1.0}, ub=P.max_imbalance_db)
        objective = {bj: 1.0 / n for bj in budgets}

    result, values = model.solve(objective)
    if not result.success or values is None:
        return np.nan
    if allocation == "uniform":
        return float(values["budget_common"])
    return float(np.mean([values[f"budget_{node}"] for node in range(n)]))


def solve_one_network(lengths, amp_counts, **kwargs):
    return np.array([
        optimize_spur_budget(lengths, int(a), **kwargs) for a in amp_counts
    ], dtype=float)


_CURVE_CACHE = {}


def ensemble_curve(amp_counts=AMP_COUNTS, **kwargs):
    """Run all 10 network MILPs and return mean and two-sided 90% CI."""
    key = (tuple(amp_counts), tuple(sorted(kwargs.items())))
    if key not in _CURVE_CACHE:
        rows = Parallel(n_jobs=N_JOBS, prefer="processes")(
            delayed(solve_one_network)(lengths, amp_counts, **kwargs)
            for lengths in NETWORKS
        )
        _CURVE_CACHE[key] = np.asarray(rows, dtype=float)

    samples = _CURVE_CACHE[key]
    count = np.sum(np.isfinite(samples), axis=0)
    mean = np.nanmean(samples, axis=0)
    std = np.nanstd(samples, axis=0, ddof=1)
    critical = np.array([
        t.ppf(0.95, c - 1) if c >= 2 else np.nan for c in count
    ])
    half_width = critical * std / np.sqrt(np.maximum(count, 1))
    return mean, mean - half_width, mean + half_width, samples


CASES = [
    (10, 27, "#e41a1c", r"$N_{Ch}=10,\ P_A=27$"),
    (10, 37, "#1769e0", r"$N_{Ch}=10,\ P_A=37$"),
    (20, 27, "#f47f31", r"$N_{Ch}=20,\ P_A=27$"),
    (20, 37, "#23abb6", r"$N_{Ch}=20,\ P_A=37$"),
]

## 4. Fig. 3 재현 — amplifier 수에 따른 평균 spur power budget

이 셀이 가장 오래 걸립니다. Colab CPU 기준 수 분 정도가 예상됩니다. 각 곡선은 10개 network realization의 평균이며 음영은 two-sided 90% confidence interval입니다.

In [ ]:
panel_defs = [
    ("balanced", "uniform", "(a) Balanced Couplers, Uniform Power Budget"),
    ("balanced", "nonuniform", "(b) Balanced Couplers, Non-uniform Power Budget"),
    ("unbalanced", "uniform", "(c) Unbalanced Couplers, Uniform Power Budget"),
    ("unbalanced", "nonuniform", "(d) Unbalanced Couplers, Non-uniform Power Budget"),
]

fig3_results = {}
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True)

for ax, (couplers, allocation, title) in zip(axes.flat, panel_defs):
    print(f"Solving Fig. 3 {title} ...")
    # The paper's balanced result is reported from 6 amplifiers; unbalanced from 4.
    plot_mask = AMP_COUNTS >= (6 if couplers == "balanced" else 4)

    for nch, pa, color, label in CASES:
        result = ensemble_curve(
            fiber="HCF", n_ch=nch, p_amp_dbm=pa,
            couplers=couplers, allocation=allocation, hybrid=False,
        )
        fig3_results[couplers, allocation, nch, pa, "HCF"] = result
        mean, low, high, _ = result
        ax.plot(AMP_COUNTS[plot_mask], mean[plot_mask], "o-", color=color, lw=2, ms=4, label=label)
        ax.fill_between(AMP_COUNTS[plot_mask], low[plot_mask], high[plot_mask], color=color, alpha=0.17)

    baseline = ensemble_curve(
        fiber="SCF", n_ch=10, p_amp_dbm=27,
        couplers=couplers, allocation=allocation, hybrid=False,
    )
    fig3_results[couplers, allocation, 10, 27, "SCF"] = baseline
    mean, low, high, _ = baseline
    baseline_mask = AMP_COUNTS >= (7 if couplers == "balanced" else 4)
    ax.plot(AMP_COUNTS[baseline_mask], mean[baseline_mask], "o--", color="black", lw=2, ms=4, label="Baseline: SCF")
    ax.fill_between(AMP_COUNTS[baseline_mask], low[baseline_mask], high[baseline_mask], color="black", alpha=0.12)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(4, 16)
    ax.set_ylim(0, 50)
    ax.set_xlabel("Number of Amplifiers")
    ax.set_ylabel("Average Power Budget [dB]")
    ax.legend(ncol=2, fontsize=8, loc="upper left")

fig.suptitle("Fig. 3 reproduction — MILP-derived power budget", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## 5. Fig. 4 재현 — Full-HCF 대비 hybrid SCF–HCF power-budget 차이

Hybrid에서는 horseshoe main links에 SCF의 $-8$ dBm/SC 비선형 상한을 적용하고, spur에는 HCF의 높은 상한을 적용합니다. 각 점은

$$\Delta P_b=P_{b,hybrid}-P_{b,full\ HCF}$$

로 계산하므로 0 dB에 가까울수록 hybrid와 full-HCF의 성능이 같다는 뜻입니다.

In [ ]:
amp_fig4 = np.arange(5, 17)
fig, axes = plt.subplots(2, 1, figsize=(8.5, 8.5), sharex=True, sharey=True)
fig4_results = {}

for ax, allocation, subtitle in zip(
    axes,
    ["uniform", "nonuniform"],
    ["(a) Uniform Power Budget", "(b) Non-uniform Power Budget"],
):
    print(f"Solving Fig. 4 {subtitle} ...")
    for nch, pa, color, label in CASES:
        full = ensemble_curve(
            amp_counts=amp_fig4, fiber="HCF", n_ch=nch, p_amp_dbm=pa,
            couplers="unbalanced", allocation=allocation, hybrid=False,
        )
        hybrid = ensemble_curve(
            amp_counts=amp_fig4, fiber="HCF", n_ch=nch, p_amp_dbm=pa,
            couplers="unbalanced", allocation=allocation, hybrid=True,
        )
        # Pair each synthetic network with itself before taking the mean.
        delta_samples = hybrid[3] - full[3]
        delta = np.nanmean(delta_samples, axis=0)
        fig4_results[allocation, nch, pa] = delta
        ax.plot(amp_fig4, delta, "o-", color=color, lw=2, ms=4, label=label)

    ax.axhline(0, color="black", lw=1)
    ax.set_title(subtitle)
    ax.set_xlim(5, 16)
    ax.set_ylim(-18, 0.5)
    ax.set_ylabel("Power Budget Difference [dB]")
    ax.legend(fontsize=8, loc="lower right")

axes[-1].set_xlabel("Number of Amplifiers")
fig.suptitle("Fig. 4 reproduction — hybrid SCF–HCF minus full HCF", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## 6. Fig. 5 재현 — spur node 수, 길이, 필요 power budget

아래 contour는 논문의 Eq. (1), (2)를 직접 평가합니다. 별도의 optimization이나 curve fitting을 사용하지 않습니다.

In [ ]:
def single_stage_budget(length_km, n_nodes, alpha=P.alpha_db_per_km):
    return alpha * length_km + 10.0 * np.log10(n_nodes)


def multi_stage_budget(length_km, n_nodes, alpha=P.alpha_db_per_km):
    return n_nodes * alpha * length_km + 10.0 * (n_nodes - 1.0) * np.log10(2.0)


fig, axes = plt.subplots(2, 1, figsize=(8.0, 10.5))

L1 = np.linspace(0.01, 80, 321)
N1 = np.linspace(1, 16, 241)
LL1, NN1 = np.meshgrid(L1, N1)
PB1 = single_stage_budget(LL1, NN1)

L2 = np.linspace(0.01, 25, 301)
N2 = np.linspace(1, 5, 241)
LL2, NN2 = np.meshgrid(L2, N2)
PB2 = multi_stage_budget(LL2, NN2)

for ax, LL, NN, PB, title, xlim, ylim in [
    (axes[0], LL1, NN1, PB1, "Single-stage Tree", (0, 80), (1, 16)),
    (axes[1], LL2, NN2, PB2, "Multi-stage Tree", (0, 25), (1, 5)),
]:
    filled = ax.contourf(LL, NN, PB, levels=np.linspace(0, 40, 21), cmap="turbo", extend="max")
    contours = ax.contour(LL, NN, PB, levels=np.arange(5, 40, 5), colors="black", linestyles="--", linewidths=1.4)
    ax.clabel(contours, inline=True, fontsize=9, fmt="%d")
    cb = fig.colorbar(filled, ax=ax, pad=0.03)
    cb.set_label(r"Power budget $P_b$ [dB]")
    ax.set(xlim=xlim, ylim=ylim, xlabel="Fiber length L [km]", ylabel="Number of nodes N")
    ax.set_title(title, weight="bold")

fig.suptitle("Fig. 5 reproduction — passive spur power-budget trade-off", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## 7. 자동 sanity check

이 검증은 논문 그래프 좌표를 넣어 맞추는 과정이 아닙니다. 공개 파라미터에서 바로 도출되는 상한과 계산 결과가 물리적으로 일관적인지 확인합니다.

In [ ]:
print("Per-SC limits selected in Fig. 2:")
for nch, pa, _, _ in CASES:
    aggregate = float(aggregate_sc_limit_dbm(nch, pa))
    hcf_cap = per_sc_cap_dbm("HCF", nch, pa)
    print(f"  NCh={nch:2d}, PA={pa:2d} dBm -> aggregate={aggregate:5.2f} dBm, HCF cap={hcf_cap:5.2f} dBm")

print(f"\nSCF theoretical spur-budget ceiling = {-8 - P.sensitivity_dbm:.1f} dB")
print(f"HCF theoretical ceiling at 10 dBm/SC = {10 - P.sensitivity_dbm:.1f} dB")

def first_saturation_amp(x, y, threshold_db=0.5):
    y = np.asarray(y)
    for i in range(1, len(y) - 1):
        if np.all(np.diff(y[i:]) < threshold_db):
            return int(x[i])
    return int(x[-1])

print("\nRepresentative saturation points from the surrogate:")
for nch, pa, _, label in CASES:
    mean = fig3_results["unbalanced", "uniform", nch, pa, "HCF"][0]
    print(f"  {label}: about {first_saturation_amp(AMP_COUNTS, mean)} amplifiers")

print("\nNotes:")
print("- Fig. 2 and Fig. 5 are equation-exact reproductions.")
print("- Fig. 3 and Fig. 4 are algorithmic MILP reproductions; no paper curve coordinates are inputs.")
print("- Residual differences mainly come from unavailable original topology samples and the unpublished full ILP implementation.")

## 참고문헌

1. M. M. Hosseini, J. Pedro, A. Napoli, “Enhanced Scalability of Horseshoe-and-Spur Networks by Exploiting Hollow-Core Fiber,” arXiv:2608.07082, 2026.
2. M. M. Hosseini *et al.*, “Optimized design of horseshoe-and-spur filterless networks leveraging point-to-multipoint coherent pluggable transceivers,” JOCN 16(10), 969–980, 2024, DOI: 10.1364/JOCN.529546.

재현 정밀도를 더 높이려면 저자들의 10개 실제 link-length 표본, 전체 node-component mapping, amplifier candidate set 및 원 ILP 코드를 확보해야 합니다.